# Mô Phỏng Máy Hút Bụi (Vacuum Cleaner Agent) – Greedy Algorithm\n\n**Quy ước ma trận:**\n- `0` → Ô trống / Máy hút bụi\n- `1` → Bụi\n- `2` → Tường / Vật cản\n\n**Chiến lược Greedy:**\n- Nếu ô hiện tại có bụi → hút ngay.\n- Nếu có ô kề chứa bụi → di chuyển đến ô đó.\n- Nếu không có bụi kề → tìm ô bụi gần nhất (Manhattan distance) và di chuyển về phía đó.

In [ ]:
import numpy as np\nimport random\nfrom collections import deque

In [ ]:
# ── Cấu hình ──\nROWS      = 5\nCOLS      = 7\nWALL_PROB = 0.15\nDUST_PROB = 0.35\nMAX_STEPS = 300\n\n# ── Tạo môi trường ngẫu nhiên ──\ndef create_env(rows, cols, wall_prob, dust_prob):\n    grid = np.zeros((rows, cols), dtype=int)\n    for r in range(rows):\n        for c in range(cols):\n            v = random.random()\n            if v < wall_prob:\n                grid[r][c] = 2\n            elif v < wall_prob + dust_prob:\n                grid[r][c] = 1\n    free = [(r, c) for r in range(rows) for c in range(cols) if grid[r][c] != 2]\n    pos = random.choice(free)\n    return grid, pos\n\ngrid, start = create_env(ROWS, COLS, WALL_PROB, DUST_PROB)\ntotal_dust = int(np.sum(grid == 1))\n\nprint(f\"Ma trận ban đầu (vị trí máy: {start}):\")\nprint(grid)\nprint(f\"Tổng bụi: {total_dust} ô\")

In [ ]:
# ── Agent (Greedy Algorithm) ──\nMOVES = {'UP': (-1,0), 'DOWN': (1,0), 'LEFT': (0,-1), 'RIGHT': (0,1)}\n\ndef bfs_distance(grid, start, target):\n    \"\"\"Tìm khoảng cách ngắn nhất từ start đến target, tránh tường (2).\"\"\"\n    rows, cols = grid.shape\n    visited = np.zeros((rows, cols), dtype=bool)\n    q = deque()\n    q.append((start[0], start[1], 0))\n    visited[start[0], start[1]] = True\n    \n    while q:\n        r, c, dist = q.popleft()\n        if (r, c) == target:\n            return dist\n        for dr, dc in MOVES.values():\n            nr, nc = r + dr, c + dc\n            if 0 <= nr < rows and 0 <= nc < cols and not visited[nr, nc] and grid[nr, nc] != 2:\n                visited[nr, nc] = True\n                q.append((nr, nc, dist + 1))\n    return float('inf')\n\ndef find_best_move(grid, pos):\n    \"\"\"\n    Greedy: chọn hướng đi đến ô bụi gần nhất.\n    1. Nếu có bụi ở ô kề → chọn ngay ô kề đó.\n    2. Ngược lại → BFS tìm ô bụi gần nhất, rồi chọn bước đầu tiên trên đường đi.\n    \"\"\"\n    rows, cols = grid.shape\n    r, c = pos\n    \n    # Bước 1: kiểm tra 4 ô kề, nếu có bụi → greedy chọn ngay\n    adj_dust = []\n    for d, (dr, dc) in MOVES.items():\n        nr, nc = r + dr, c + dc\n        if 0 <= nr < rows and 0 <= nc < cols and grid[nr, nc] == 1:\n            adj_dust.append((d, dr, dc))\n    if adj_dust:\n        return random.choice(adj_dust)  # nếu nhiều ô kề có bụi, chọn ngẫu nhiên 1\n    \n    # Bước 2: tìm ô bụi gần nhất bằng BFS\n    dust_cells = [(dr, dc) for dr in range(rows) for dc in range(cols) if grid[dr, dc] == 1]\n    if not dust_cells:\n        return None  # không còn bụi\n    \n    # Tìm ô bụi gần nhất\n    best_dist = float('inf')\n    best_target = None\n    for target in dust_cells:\n        d = bfs_distance(grid, pos, target)\n        if d < best_dist:\n            best_dist = d\n            best_target = target\n    \n    if best_target is None or best_dist == float('inf'):\n        return None\n    \n    # Từ vị trí hiện tại, chọn bước đi đầu tiên về phía best_target\n    # Dùng BFS để lấy đường đi ngắn nhất\n    visited = np.zeros((rows, cols), dtype=bool)\n    parent = {}\n    q = deque()\n    q.append((r, c))\n    visited[r, c] = True\n    \n    found = False\n    while q and not found:\n        cr, cc = q.popleft()\n        for d, (dr, dc) in MOVES.items():\n            nr, nc = cr + dr, cc + dc\n            if 0 <= nr < rows and 0 <= nc < cols and not visited[nr, nc] and grid[nr, nc] != 2:\n                visited[nr, nc] = True\n                parent[(nr, nc)] = (cr, cc, d)\n                if (nr, nc) == best_target:\n                    found = True\n                    break\n                q.append((nr, nc))\n    \n    # Truy ngược để lấy bước đi đầu tiên\n    cur = best_target\n    while cur in parent:\n        pr, pc, direction = parent[cur]\n        if (pr, pc) == (r, c):\n            dr, dc = MOVES[direction]\n            return (direction, dr, dc)\n        cur = (pr, pc)\n    \n    return None\n\ndef run_agent(grid_in, start, max_steps):\n    grid = grid_in.copy()\n    rows, cols = grid.shape\n    pos = list(start)\n    history = set()\n    steps = 0\n    cleaned = 0\n    \n    while steps < max_steps:\n        r, c = pos\n        \n        # Nếu ô hiện tại có bụi → hút\n        if grid[r][c] == 1:\n            state = ((r, c), 'CLEAN')\n            if state in history:\n                return grid, steps, cleaned, 'THAT BAI', 'Lap lai hanh dong CLEAN tai ' + str((r, c))\n            history.add(state)\n            grid[r][c] = 0\n            cleaned += 1\n            steps += 1\n            if cleaned == total_dust:\n                return grid, steps, cleaned, 'THANH CONG', 'Da hut sach toan bo bui'\n            continue\n        \n        # Greedy: chọn hướng đi tốt nhất\n        best = find_best_move(grid, pos)\n        if best is None:\n            # Không còn bụi hoặc không tìm được đường\n            if cleaned == total_dust:\n                return grid, steps, cleaned, 'THANH CONG', 'Da hut sach toan bo bui'\n            return grid, steps, cleaned, 'THAT BAI', 'Khong tim thay duong di den bui'\n        \n        direction, dr, dc = best\n        state = ((r, c), direction)\n        if state in history:\n            return grid, steps, cleaned, 'THAT BAI', f'Lap lai hanh dong {direction} tai {(r, c)}'\n        history.add(state)\n        \n        pos = [r + dr, c + dc]\n        steps += 1\n    \n    return grid, steps, cleaned, 'THAT BAI', f'Vuot qua gioi han {max_steps} buoc'\n\n\nfinal_grid, steps, cleaned, status, reason = run_agent(grid, start, MAX_STEPS)\n\nprint(\"Ma tran sau khi chay:\")\nprint(final_grid)\nprint()\nprint(f\"So buoc di : {steps}\")\nprint(f\"Bui da hut : {cleaned} / {total_dust} o\")\nprint(f\"Trang thai : {status}\")\nprint(f\"Ly do      : {reason}\")